# EV Charging Points Visualization

This notebook visualizes the distribution of Electric Vehicle (EV) charging stations across Spain.

In [ ]:
import polars as pl
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import os

## 1. Load data

### 1.1 Load Charging Points data

In [ ]:
data_path = "../data/processed/charging_points.parquet"
charging_points = pl.read_parquet(data_path)
charging_points

In [ ]:
charging_points = charging_points.group_by([pl.col("site_id"),pl.col('latitude'),pl.col('longitude')]).agg([
    pl.col("site_name").first(),
    pl.col("connector_type").unique(),
    pl.col("connector_type").count().alias('n_chargers'),
    pl.col("max_power").unique(),
    pl.col("charging_mode").unique()
]).filter(
    (pl.col("latitude").is_not_null()) & (pl.col("longitude").is_not_null())
)

print(f"Grouped into {charging_points.height} unique charging sites.")
charging_points.head()

In [ ]:
data_path = "../data/processed/charging_stations.parquet"
charging_stations = gpd.read_parquet(data_path)
charging_stations = charging_stations.to_crs(epsg=4326)
charging_stations

### 1.3 Load Road Segments

In [ ]:
import geopandas as gpd
# Paths
processed_path = '../data/processed/integrated_road_network.parquet'

gdf_processed = gpd.read_parquet(processed_path)

if gdf_processed.crs != 4326:
    gdf_processed = gdf_processed.to_crs(epsg=4326)

print(f"Loaded {len(gdf_processed)} Road Segments.")
gdf_processed.head()

## 2. Interactive Maps

We use **CartoDB Positron** tiles for a clean visual style. Below are two ways to visualize the EV charging infrastructure:

1. **Ultra Fast Charging Points and Road Segments**
2. **Ultra Fast Charging Points near Road Segments (<1km)**

### 2.1 Ultra Fast Charging Points and Road Segments

In [ ]:
# Individual Points Visualization (No Clustering)
m_roads_charging_points = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles="CartoDB Positron")

for row in charging_points.to_dicts():
    # Use CircleMarker for better performance and cleaner look when not clustering
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color="#3186cc",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.7,
        tooltip=row['site_name']
    ).add_to(m_roads_charging_points)


# Add Processed Segments
folium.GeoJson(
    gdf_processed,
    name="Processed Network",
    style_function=lambda x: {'color': 'blue', 'weight': 2, 'opacity': 0.7},
).add_to(m_roads_charging_points)

m_roads_charging_points

### 2.2 Ultra Fast Charging Points 1km or closer to Road Segments

In [ ]:
# Individual Points Visualization (No Clustering)
m_roads_charging_stations = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles="CartoDB Positron")

for row in charging_stations.itertuples():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color="#3186cc",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.7,
        tooltip=getattr(row, "site_name", "")
    ).add_to(m_roads_charging_stations)


# Add Processed Segments
folium.GeoJson(
    gdf_processed,
    name="Processed Network",
    style_function=lambda x: {'color': 'blue', 'weight': 2, 'opacity': 0.7},
).add_to(m_roads_charging_stations)

m_roads_charging_stations